# Holiday Calendar (Idiomatic Notebook)

This notebook uses the well-supported `holidays` package to generate holiday dates with a concise, notebook-first workflow.

In [19]:
# Notebook parameters
start_year = 2024
end_year = 2028
country = "US"
subdiv = None  # Example: "CA" for California

In [20]:
# Install once per environment if needed
%pip -q install holidays ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [21]:
import json
import pandas as pd
import holidays
import ipywidgets as widgets
from IPython.display import Javascript, display

In [22]:
# Build a tidy holiday table
years = range(start_year, end_year + 1)
cal = holidays.country_holidays(country, subdiv=subdiv, years=years, observed=True)

df = (
    pd.Series(cal, name="Holiday")
    .rename_axis("Date")
    .sort_index()
    .reset_index()
)

df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year
df["Weekday"] = df["Date"].dt.day_name()

df.head(10)

,Date,Holiday,Year,Weekday
0,2024-01-01,New Year's Day,2024,Monday
1,2024-01-15,Martin Luther King Jr. Day,2024,Monday
2,2024-02-19,Washington's Birthday,2024,Monday
3,2024-05-27,Memorial Day,2024,Monday
4,2024-06-19,Juneteenth National Independence Day,2024,Wednesday
5,2024-07-04,Independence Day,2024,Thursday
6,2024-09-02,Labor Day,2024,Monday
7,2024-10-14,Columbus Day,2024,Monday
8,2024-11-11,Veterans Day,2024,Monday
9,2024-11-28,Thanksgiving Day,2024,Thursday


In [23]:
# Compact yearly summary
summary = (
    df.groupby("Year", as_index=False)
    .agg(Holiday_Count=("Holiday", "count"))
)
summary

,Year,Holiday_Count
0,2024,11
1,2025,11
2,2026,12
3,2027,15
4,2028,12


In [24]:
# Inspect one year
year = 2026
df.loc[df["Year"] == year, ["Date", "Weekday", "Holiday"]].reset_index(drop=True)

,Date,Weekday,Holiday
0,2026-01-01,Thursday,New Year's Day
1,2026-01-19,Monday,Martin Luther King Jr. Day
2,2026-02-16,Monday,Washington's Birthday
3,2026-05-25,Monday,Memorial Day
4,2026-06-19,Friday,Juneteenth National Independence Day
5,2026-07-03,Friday,Independence Day (observed)
6,2026-07-04,Saturday,Independence Day
7,2026-09-07,Monday,Labor Day
8,2026-10-12,Monday,Columbus Day
9,2026-11-11,Wednesday,Veterans Day


## Quick Validation Tests

Each test is a single code cell that shows expected vs actual values as notebook output.
Assertions enforce correctness without print-based status messages.

In [25]:
# ApprovalTest type and helpers (MIME comes from rich repr; actions use ipywidgets callbacks)
import html

APPROVAL_LAST_ACTUAL = {}

def to_iso_records(frame):
    normalized = frame.copy()
    for col in normalized.columns:
        if pd.api.types.is_datetime64_any_dtype(normalized[col]):
            normalized[col] = normalized[col].dt.strftime("%Y-%m-%d")
    return normalized.to_dict("records")

def stable_records(records, sort_by):
    if not sort_by:
        return records
    frame = pd.DataFrame(records)
    return frame.sort_values(sort_by).reset_index(drop=True).to_dict("records")

def _approval_storage_source():
    return "\n".join([
        "# APPROVAL_STORAGE",
        f"APPROVAL_BASELINES = {json.dumps(APPROVAL_BASELINES, indent=4, sort_keys=True)}",
        f"APPROVAL_DECISIONS = {json.dumps(APPROVAL_DECISIONS, indent=4, sort_keys=True)}",
    ])

def _replace_storage_cell_via_frontend():
    source = _approval_storage_source()
    js_payload = json.dumps(source)
    display(Javascript(
        """
(function() {
  const marker = '# APPROVAL_STORAGE';
  const newSource = """ + js_payload + """;

  if (window.Jupyter && Jupyter.notebook && Jupyter.notebook.get_cells) {
    const cells = Jupyter.notebook.get_cells();
    for (const cell of cells) {
      if (typeof cell.get_text === 'function' && cell.get_text().trimStart().startsWith(marker)) {
        cell.set_text(newSource);
        return;
      }
    }
  }

  if (window.vscodeNotebookCellModificationHelpers && typeof window.vscodeNotebookCellModificationHelpers.replaceCellSource === 'function') {
    window.vscodeNotebookCellModificationHelpers.replaceCellSource(marker, newSource);
  }
})();
"""
    ))

class ApprovalTest:
    def __init__(self, test_id, description, actual, approved=None, decision=None, context=None):
        self.test_id = test_id
        self.description = description
        self.context = context or {"@vocab": "https://schema.org/"}
        self.actual = actual
        self.approved = approved
        self.decision = decision

        if decision == "disapproved":
            self.passed = False
            self.status = "disapproved"
            self.diff = {"approved": approved, "actual": actual}
        elif approved is None:
            self.passed = False
            self.status = "missing-approved"
            self.diff = {"approved": None, "actual": actual}
        elif approved == actual:
            self.passed = True
            self.status = "approved"
            self.diff = []
        else:
            self.passed = False
            self.status = "changed"
            self.diff = {"approved": approved, "actual": actual}

    def _jsonld_payload(self):
        return {
            "@context": self.context,
            "@type": "PropertyValue",
            "name": self.test_id,
            "description": self.description,
            "value": self.actual,
            "expectedValue": self.approved,
            "result": self.passed,
            "additionalProperty": [
                {"@type": "PropertyValue", "name": "status", "value": self.status},
                {"@type": "PropertyValue", "name": "decision", "value": self.decision},
                {"@type": "PropertyValue", "name": "diff", "value": self.diff},
            ],
        }

    def _html_view(self):
        actual_text = html.escape(json.dumps(self.actual, indent=2, ensure_ascii=True))
        approved_text = html.escape(json.dumps(self.approved, indent=2, ensure_ascii=True))
        status_color = "#0f766e" if self.passed else "#b91c1c"
        border_color = "#14b8a6" if self.passed else "#ef4444"

        return f"""
<div style='border:1px solid {border_color}; border-radius:10px; padding:12px; margin:8px 0; font-family:ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;'>
  <div style='display:flex; justify-content:space-between; align-items:center; gap:8px;'>
    <div>
      <strong>{html.escape(self.test_id)}</strong><br/>
      <span style='font-size:12px; color:#334155;'>{html.escape(self.description)}</span>
    </div>
    <span style='color:{status_color}; font-weight:700;'>{html.escape(self.status)}</span>
  </div>
  <details style='margin-top:10px;' open>
    <summary><strong>Approved (from storage cell)</strong></summary>
    <pre style='white-space:pre-wrap; background:#f8fafc; padding:8px; border-radius:6px;'>{approved_text}</pre>
  </details>
  <details style='margin-top:10px;' open>
    <summary><strong>Actual</strong></summary>
    <pre style='white-space:pre-wrap; background:#f8fafc; padding:8px; border-radius:6px;'>{actual_text}</pre>
  </details>
</div>
"""

    def _repr_mimebundle_(self, include=None, exclude=None):
        payload = self._jsonld_payload()
        data = {
            "application/ld+json": payload,
            "application/json": payload,
            "text/html": self._html_view(),
            "text/plain": json.dumps(payload, indent=2, ensure_ascii=True),
        }
        metadata = {
            "application/json": {"expanded": True},
            "application/ld+json": {
                "expanded": True,
                "approvalTest": {
                    "testId": self.test_id,
                    "status": self.status,
                    "decision": self.decision,
                    "approved": self.approved,
                },
            },
        }
        return data, metadata

def approval_action(action, test_id):
    if action not in {"approve", "disapprove"}:
        raise ValueError("action must be 'approve' or 'disapprove'")
    if test_id not in APPROVAL_LAST_ACTUAL:
        raise KeyError(f"No actual value registered for test: {test_id}")

    if action == "approve":
        APPROVAL_BASELINES[test_id] = APPROVAL_LAST_ACTUAL[test_id]
        APPROVAL_DECISIONS[test_id] = "approved"
    else:
        APPROVAL_DECISIONS[test_id] = "disapproved"

    _replace_storage_cell_via_frontend()

def approval_controls(test_id):
    approve_btn = widgets.Button(description="Approve", button_style="success", icon="check")
    disapprove_btn = widgets.Button(description="Disapprove", button_style="danger", icon="times")
    status = widgets.HTML(value="<small>Use buttons to update storage cell state.</small>")

    def on_approve(_):
        approval_action("approve", test_id)
        status.value = "<small><b>Approved.</b> Storage updated in memory and cell source refresh requested.</small>"

    def on_disapprove(_):
        approval_action("disapprove", test_id)
        status.value = "<small><b>Disapproved.</b> Test is forced to fail until approved again.</small>"

    approve_btn.on_click(on_approve)
    disapprove_btn.on_click(on_disapprove)

    return widgets.VBox([
        widgets.HBox([approve_btn, disapprove_btn]),
        status,
    ])

def run_approval_test(test_id, description, actual, sort_by=None):
    actual_sorted = stable_records(actual, sort_by) if isinstance(actual, list) else actual
    APPROVAL_LAST_ACTUAL[test_id] = actual_sorted
    approved = APPROVAL_BASELINES.get(test_id)
    decision = APPROVAL_DECISIONS.get(test_id)
    return ApprovalTest(
        test_id=test_id,
        description=description,
        actual=actual_sorted,
        approved=approved,
        decision=decision,
    )

def show_approval_test(test_id, description, actual, sort_by=None):
    result = run_approval_test(test_id, description, actual, sort_by=sort_by)
    display(result)
    display(approval_controls(test_id))
    return None

def approval_from_dataframe(test_id, description, actual_df, sort_by=None):
    actual_records = to_iso_records(actual_df)
    return show_approval_test(test_id, description, actual_records, sort_by=sort_by)

In [26]:
# APPROVAL_STORAGE
APPROVAL_BASELINES = {
    "required_columns_present": {
        "columns": ["Date", "Holiday", "Weekday", "Year"]
    },
    "year_boundaries": {
        "startYear": 2024,
        "endYear": 2028
    },
    "date_order_and_weekday_derivation": {
        "datesSorted": True,
        "weekdayMatchesDate": True
    },
    "known_holiday_spot_checks": [
        {"Date": "2024-01-01", "Expected": "New Year's Day"},
        {"Date": "2024-11-28", "Expected": "Thanksgiving Day"},
        {"Date": "2026-05-25", "Expected": "Memorial Day"}
    ]
}
APPROVAL_DECISIONS = {}

In [27]:
# Test 1: required columns are present (one-cell test)
actual_columns = sorted(df.columns.tolist())
required = ["Date", "Holiday", "Weekday", "Year"]
missing = [c for c in required if c not in actual_columns]
assert not missing, f"Missing columns: {missing}"

show_approval_test(
    test_id="required_columns_present",
    description="Dataframe includes required columns.",
    actual={"columns": actual_columns},
)

{
  "@context": {
    "@vocab": "https://schema.org/"
  },
  "@type": "PropertyValue",
  "name": "required_columns_present",
  "description": "Dataframe includes required columns.",
  "value": {
    "columns": [
      "Date",
      "Holiday",
      "Weekday",
      "Year"
    ]
  },
  "expectedValue": {
    "columns": [
      "Date",
      "Holiday",
      "Weekday",
      "Year"
    ]
  },
  "result": true,
  "additionalProperty": [
    {
      "@type": "PropertyValue",
      "name": "status",
      "value": "approved"
    },
    {
      "@type": "PropertyValue",
      "name": "decision",
      "value": null
    },
    {
      "@type": "PropertyValue",
      "name": "diff",
      "value": []
    }
  ]
}

In [28]:
# Test 2: year boundaries are correct (one-cell test)
actual_min = int(df["Year"].min())
actual_max = int(df["Year"].max())
assert actual_min == start_year, f"Start year mismatch: expected {start_year}, got {actual_min}"
assert actual_max == end_year, f"End year mismatch: expected {end_year}, got {actual_max}"

show_approval_test(
    test_id="year_boundaries",
    description="Observed year range matches configured notebook parameters.",
    actual={"startYear": actual_min, "endYear": actual_max},
)

{
  "@context": {
    "@vocab": "https://schema.org/"
  },
  "@type": "PropertyValue",
  "name": "year_boundaries",
  "description": "Observed year range matches configured notebook parameters.",
  "value": {
    "startYear": 2024,
    "endYear": 2028
  },
  "expectedValue": {
    "startYear": 2024,
    "endYear": 2028
  },
  "result": true,
  "additionalProperty": [
    {
      "@type": "PropertyValue",
      "name": "status",
      "value": "approved"
    },
    {
      "@type": "PropertyValue",
      "name": "decision",
      "value": null
    },
    {
      "@type": "PropertyValue",
      "name": "diff",
      "value": []
    }
  ]
}

In [29]:
# Test 3: date ordering and weekday derivation (one-cell test)
is_sorted = bool(df["Date"].is_monotonic_increasing)
weekday_match = bool((df["Date"].dt.day_name() == df["Weekday"]).all())
assert is_sorted, "Dates are not sorted ascending"
assert weekday_match, "Weekday values do not match Date"

show_approval_test(
    test_id="date_order_and_weekday_derivation",
    description="Dates are sorted and weekday text matches derived weekday names.",
    actual={"datesSorted": is_sorted, "weekdayMatchesDate": weekday_match},
)

{
  "@context": {
    "@vocab": "https://schema.org/"
  },
  "@type": "PropertyValue",
  "name": "date_order_and_weekday_derivation",
  "description": "Dates are sorted and weekday text matches derived weekday names.",
  "value": {
    "datesSorted": true,
    "weekdayMatchesDate": true
  },
  "expectedValue": {
    "datesSorted": true,
    "weekdayMatchesDate": true
  },
  "result": true,
  "additionalProperty": [
    {
      "@type": "PropertyValue",
      "name": "status",
      "value": "approved"
    },
    {
      "@type": "PropertyValue",
      "name": "decision",
      "value": null
    },
    {
      "@type": "PropertyValue",
      "name": "diff",
      "value": []
    }
  ]
}

In [30]:
# Test 4: known holiday spot checks (one-cell test)
known = pd.DataFrame([
    {"Date": pd.Timestamp("2024-01-01"), "Expected": "New Year's Day"},
    {"Date": pd.Timestamp("2024-11-28"), "Expected": "Thanksgiving Day"},
    {"Date": pd.Timestamp("2026-05-25"), "Expected": "Memorial Day"},
])

actual = (
    df.merge(known[["Date"]], on="Date", how="inner")
      .groupby("Date", as_index=False)
      .agg(Expected=("Holiday", lambda s: " | ".join(sorted(set(s)))))
)
actual_records = to_iso_records(actual)

show_approval_test(
    test_id="known_holiday_spot_checks",
    description="Known holiday checkpoints match expected names for specific dates.",
    actual=actual_records,
    sort_by=["Date", "Expected"],
)

{
  "@context": {
    "@vocab": "https://schema.org/"
  },
  "@type": "PropertyValue",
  "name": "known_holiday_spot_checks",
  "description": "Known holiday checkpoints match expected names for specific dates.",
  "value": [
    {
      "Date": "2024-01-01",
      "Expected": "New Year's Day"
    },
    {
      "Date": "2024-11-28",
      "Expected": "Thanksgiving Day"
    },
    {
      "Date": "2026-05-25",
      "Expected": "Memorial Day"
    }
  ],
  "expectedValue": [
    {
      "Date": "2024-01-01",
      "Expected": "New Year's Day"
    },
    {
      "Date": "2024-11-28",
      "Expected": "Thanksgiving Day"
    },
    {
      "Date": "2026-05-25",
      "Expected": "Memorial Day"
    }
  ],
  "result": true,
  "additionalProperty": [
    {
      "@type": "PropertyValue",
      "name": "status",
      "value": "approved"
    },
    {
      "@type": "PropertyValue",
      "name": "decision",
      "value": null
    },
    {
      "@type": "PropertyValue",
      "name": "diff",
      "value": []
    }
  ]
}

## Concise Approval Tests (Visible Approved vs Actual)

This style keeps approvals directly in notebook cells.
Each test cell displays Approved and Actual values, then fails with a focused diff when they differ.

### Notebook Testing Patterns In Practice

- ipytest: run pytest directly inside notebook cells via magics.
- nbval or pytest-notebook: rerun whole notebooks and compare stored outputs, usually in CI.
- testbook: write tests in separate Python files against notebook code.
- ApprovalTests.Python: classic approved vs received workflow with diff tools.

This notebook uses per-cell approved tables so each test visibly shows Approved and Actual values in one place.

In [31]:
# Helper note: JSON-LD helpers are defined above the tests to support linear top-to-bottom execution.
# This cell is intentionally left minimal to avoid redefining hooks later in the notebook.

In [32]:
# Approval test 1: full 2026 holiday list (approved baseline stored in storage cell)
actual_2026 = df.loc[df["Year"] == 2026, ["Date", "Holiday"]]
approval_from_dataframe(
    test_id="approval_us_2026_holidays",
    description="Federal holidays for 2026 (including observed).",
    actual_df=actual_2026,
    sort_by=["Date", "Holiday"],
)

{
  "@context": {
    "@vocab": "https://schema.org/"
  },
  "@type": "PropertyValue",
  "name": "approval_us_2026_holidays",
  "description": "Federal holidays for 2026 (including observed).",
  "value": [
    {
      "Date": "2026-01-01",
      "Holiday": "New Year's Day"
    },
    {
      "Date": "2026-01-19",
      "Holiday": "Martin Luther King Jr. Day"
    },
    {
      "Date": "2026-02-16",
      "Holiday": "Washington's Birthday"
    },
    {
      "Date": "2026-05-25",
      "Holiday": "Memorial Day"
    },
    {
      "Date": "2026-06-19",
      "Holiday": "Juneteenth National Independence Day"
    },
    {
      "Date": "2026-07-03",
      "Holiday": "Independence Day (observed)"
    },
    {
      "Date": "2026-07-04",
      "Holiday": "Independence Day"
    },
    {
      "Date": "2026-09-07",
      "Holiday": "Labor Day"
    },
    {
      "Date": "2026-10-12",
      "Holiday": "Columbus Day"
    },
    {
      "Date": "2026-11-11",
      "Holiday": "Veterans Day"
    },
    {
      "Date": "2026-11-26",
      "Holiday": "Thanksgiving Day"
    },
    {
      "Date": "2026-12-25",
      "Holiday": "Christmas Day"
    }
  ],
  "expectedValue": null,
  "result": false,
  "additionalProperty": [
    {
      "@type": "PropertyValue",
      "name": "status",
      "value": "missing-approved"
    },
    {
      "@type": "PropertyValue",
      "name": "decision",
      "value": null
    },
    {
      "@type": "PropertyValue",
      "name": "diff",
      "value": {
        "approved": null,
        "actual": [
          {
            "Date": "2026-01-01",
            "Holiday": "New Year's Day"
          },
          {
            "Date": "2026-01-19",
            "Holiday": "Martin Luther King Jr. Day"
          },
          {
            "Date": "2026-02-16",
            "Holiday": "Washington's Birthday"
          },
          {
            "Date": "2026-05-25",
            "Holiday": "Memorial Day"
          },
          {
            "Date": "2026-06-19",
            "Holiday": "Juneteenth National Independence Day"
          },
          {
            "Date": "2026-07-03",
            "Holiday": "Independence Day (observed)"
          },
          {
            "Date": "2026-07-04",
            "Holiday": "Independence Day"
          },
          {
            "Date": "2026-09-07",
            "Holiday": "Labor Day"
          },
          {
            "Date": "2026-10-12",
            "Holiday": "Columbus Day"
          },
          {
            "Date": "2026-11-11",
            "Holiday": "Veterans Day"
          },
          {
            "Date": "2026-11-26",
            "Holiday": "Thanksgiving Day"
          },
          {
            "Date": "2026-12-25",
            "Holiday": "Christmas Day"
          }
        ]
      }
    }
  ]
}

In [33]:
# Approval test 2: Independence Day observed behavior in 2026
actual_observed = df.loc[
    (df["Date"].between(pd.Timestamp("2026-07-03"), pd.Timestamp("2026-07-04")))
    & (df["Holiday"].str.contains("Independence Day")),
    ["Date", "Holiday"],
]
approval_from_dataframe(
    test_id="approval_independence_day_observed_2026",
    description="Observed Independence Day appears on Friday when July 4 is Saturday.",
    actual_df=actual_observed,
    sort_by=["Date", "Holiday"],
)

{
  "@context": {
    "@vocab": "https://schema.org/"
  },
  "@type": "PropertyValue",
  "name": "approval_independence_day_observed_2026",
  "description": "Observed Independence Day appears on Friday when July 4 is Saturday.",
  "value": [
    {
      "Date": "2026-07-03",
      "Holiday": "Independence Day (observed)"
    },
    {
      "Date": "2026-07-04",
      "Holiday": "Independence Day"
    }
  ],
  "expectedValue": null,
  "result": false,
  "additionalProperty": [
    {
      "@type": "PropertyValue",
      "name": "status",
      "value": "missing-approved"
    },
    {
      "@type": "PropertyValue",
      "name": "decision",
      "value": null
    },
    {
      "@type": "PropertyValue",
      "name": "diff",
      "value": {
        "approved": null,
        "actual": [
          {
            "Date": "2026-07-03",
            "Holiday": "Independence Day (observed)"
          },
          {
            "Date": "2026-07-04",
            "Holiday": "Independence Day"
          }
        ]
      }
    }
  ]
}